# Hierarchical Probabilistic U-Net: spatial latent interventions

The [hierarchical Probabilistic U-Net](https://arxiv.org/abs/1905.13077) conditions each finer latent distribution on the coarser samples. This tutorial uses an SMP encoder and the toolbox toy segmentation data, with no pretrained-weight download. It demonstrates training, prior sampling, and per-scale interventions. A brief toy run cannot establish the scale disentanglement or LIDC accuracy reported in the paper.


In [ ]:
# Copyright (c) 2023 lightning-uq-box. All rights reserved.
# Licensed under the Apache License 2.0.
from functools import partial

import matplotlib.pyplot as plt
import torch
from lightning import Trainer, seed_everything

from lightning_uq_box.datamodules import ToySegmentationDataModule
from lightning_uq_box.models import HierarchicalProbUNet as HierarchicalModel
from lightning_uq_box.uq_methods import HierarchicalProbUNet

seed_everything(0)
torch.set_num_threads(2)
data = ToySegmentationDataModule(batch_size=4)
model = HierarchicalModel(
    encoder_name="resnet18",
    encoder_weights=None,
    classes=4,
    latent_dims=(1, 1),
    decoder_channels=(32, 16, 8, 4, 2),
)
method = HierarchicalProbUNet(
    model, num_classes=4, num_samples=4, optimizer=partial(torch.optim.Adam, lr=1e-4)
)
trainer = Trainer(
    max_steps=20,
    accelerator="cpu",
    devices=1,
    logger=False,
    enable_checkpointing=False,
    gradient_clip_val=1.0,
    limit_val_batches=1,
    enable_model_summary=False,
)
trainer.fit(method, datamodule=data)

## Sample segmentations and inspect uncertainty

Training uses posterior samples and evaluates the prior on those same samples. At prediction time, only the image is available and all samples come from the prior. `pred` is the mean probability map; `logits` retains each sample.


In [ ]:
method.eval()
batch = next(iter(data.val_dataloader()))
image = batch["input"][:1]
with torch.no_grad():
    predictions = method.predict_step(image)
print("Sample logits:", tuple(predictions["logits"].shape))
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(image[0].movedim(0, -1).numpy())
axes[0].set_title("Input")
axes[1].imshow(predictions["pred"].argmax(1)[0], vmin=0, vmax=3)
axes[1].set_title("Mean prediction")
axes[2].imshow(predictions["pred_uct"][0].squeeze(), cmap="magma")
axes[2].set_title("Predictive entropy")
for ax in axes:
    ax.axis("off")
plt.show()
plt.close(fig)

## Intervene at one scale

First fix every latent at its conditional prior mean. In each row below, resample one scale while using conditional means at the others. Because the hierarchy is autoregressive, a coarse intervention also changes the means of downstream distributions. Use the `z_q` argument with saved latent tensors instead if you need to hold their numeric values fixed.

For a trained LIDC model, inspect whether coarse changes affect lesion presence or structure and fine changes affect boundaries. Also inspect per-scale `train_kl_i`, `train_lagmul`, and held-out reconstruction IoU; visually different samples alone do not establish disentanglement.


In [ ]:
scales = len(model.latent_dims)
fig, axes = plt.subplots(scales, 5, figsize=(12, 3 * scales), squeeze=False)
with torch.no_grad():
    baseline = model.sample(image, mean=True).argmax(1)[0]
    for scale in range(scales):
        flags = [True] * scales
        flags[scale] = False
        axes[scale, 0].imshow(baseline, vmin=0, vmax=3)
        axes[scale, 0].set_title("All means")
        for col in range(1, 5):
            sample = model.sample(image, mean=flags).argmax(1)[0]
            axes[scale, col].imshow(sample, vmin=0, vmax=3)
            axes[scale, col].set_title(f"Scale {scale}, draw {col}")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()
plt.close(fig)

## Evaluate multiple annotations

Given integer prior maps `samples` of shape `[B, N, H, W]` and grader maps `graders` of shape `[B, M, H, W]`, call `generalized_energy_distance(samples, graders, num_classes)` and `hungarian_matched_iou(samples, graders, num_classes)` from `lightning_uq_box.eval_utils`. Report GED squared together with `d_ss` and matched IoU. Empty foreground in both masks scores IoU 1.

For LIDC, use the same SMP backbone and preprocessing as the flat baseline. The flat model uses its own beta-ELBO loss while this model defaults to GECO. These are a controlled backbone comparison, not an exact replication of the paper's original networks.
